# Prediction and Post-processing
Applies the trained 4-channel model to a full mosaic and cleans the output through
nodata masking, NDVI filtering, built-up masking, and sieve filtering. Prediction uses
tiled blocks and post-processing is fully file-based, so the notebook stays within
Colab memory limits on large AOIs.

**Runtime:** T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y gdal-bin > /dev/null 2>&1
!pip install segmentation-models-pytorch rasterio scikit-image -q

In [ ]:
import os, glob, shutil, subprocess
import numpy as np
import torch
import segmentation_models_pytorch as smp
import rasterio
from rasterio.windows import Window
from skimage.filters import threshold_otsu
from osgeo import gdal

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

In [ ]:
TILES_DIR = '/content/drive/MyDrive/asm-prediction/images/2025'
BUILTUP_PATH = '/content/drive/MyDrive/asm-v2/masks/builtup-mask.tif'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/asm-v2/footprints'
MODEL_PATH = '/content/drive/MyDrive/asm-v2/predict_model/asm_unet_v1.pth'

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

ENCODER = 'resnet34'
IN_CHANNELS = 4
PATCH_SIZE = 256
OVERLAP = 64

TILE_SIZE = 10000      # prediction block size in pixels
TILE_BUFFER = 256      # overlap buffer between blocks to avoid edge artefacts

MIN_PATCH_PIXELS = 50  # sieve threshold: clusters smaller than this are removed
STRIP_HEIGHT = 2048    # strip height for post-processing I/O

print('Configuration loaded')

## Build mosaic and load model

In [ ]:
# Build VRT if multiple tiles, otherwise use the single file directly
tiles = sorted(glob.glob(os.path.join(TILES_DIR, '*.tif')))
print(f'Found {len(tiles)} tile(s)')

VRT_PATH = os.path.join(WORK_DIR, 'mosaic.vrt')

if len(tiles) == 1:
    VRT_PATH = tiles[0]
    print(f'Single file: {VRT_PATH}')
else:
    subprocess.run(['gdalbuildvrt', VRT_PATH] + tiles, check=True)
    print(f'VRT built: {VRT_PATH}')

with rasterio.open(VRT_PATH) as src:
    FULL_HEIGHT, FULL_WIDTH = src.height, src.width
    FULL_PROFILE = src.profile.copy()
    print(f'Dimensions: {FULL_WIDTH} x {FULL_HEIGHT}')
    print(f'Bands: {src.count}')

In [ ]:
model = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=IN_CHANNELS, classes=1)
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model = model.to(device)
model.eval()
print('Model loaded')

## Tiled prediction
The mosaic is split into blocks. Each block is read with a buffer on all sides,
predicted with a sliding window, then the buffer is trimmed and the result written
to the output file. Only one block is in memory at a time.

In [ ]:
def predict_block(src, model, y_start, x_start, block_h, block_w, buffer):
    """Predict a single block with buffer overlap. Returns clipped binary result."""
    buf_y = max(0, y_start - buffer)
    buf_x = max(0, x_start - buffer)
    buf_y_end = min(FULL_HEIGHT, y_start + block_h + buffer)
    buf_x_end = min(FULL_WIDTH, x_start + block_w + buffer)
    buf_h = buf_y_end - buf_y
    buf_w = buf_x_end - buf_x

    stride = PATCH_SIZE - OVERLAP
    pred_sum = np.zeros((buf_h, buf_w), dtype=np.float32)
    pred_count = np.zeros((buf_h, buf_w), dtype=np.float32)

    with torch.no_grad():
        for py in range(0, buf_h - PATCH_SIZE + 1, stride):
            for px in range(0, buf_w - PATCH_SIZE + 1, stride):
                window = Window(buf_x + px, buf_y + py, PATCH_SIZE, PATCH_SIZE)
                img = src.read(window=window).astype(np.float32)
                img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
                img = np.clip(img / 10000.0, 0, 1)  # Planet NICFI reflectance to 0-1

                tensor = torch.tensor(img).unsqueeze(0).to(device)
                pred = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()

                pred_sum[py:py+PATCH_SIZE, px:px+PATCH_SIZE] += pred
                pred_count[py:py+PATCH_SIZE, px:px+PATCH_SIZE] += 1

    pred_count[pred_count == 0] = 1
    pred_avg = pred_sum / pred_count

    # Trim the buffer to get just the target block
    clip_y = y_start - buf_y
    clip_x = x_start - buf_x
    result = pred_avg[clip_y:clip_y+block_h, clip_x:clip_x+block_w]

    return (result > 0.5).astype(np.uint8)

In [ ]:
RAW_PATH = os.path.join(WORK_DIR, 'galamsey_prediction_raw.tif')

out_profile = FULL_PROFILE.copy()
out_profile.update(driver='GTiff', count=1, dtype='uint8', compress='lzw', nodata=None)

n_blocks_y = len(range(0, FULL_HEIGHT, TILE_SIZE))
n_blocks_x = len(range(0, FULL_WIDTH, TILE_SIZE))
total_blocks = n_blocks_y * n_blocks_x
print(f'Grid: {n_blocks_x} x {n_blocks_y} = {total_blocks} blocks')

with rasterio.open(VRT_PATH) as src:
    with rasterio.open(RAW_PATH, 'w', **out_profile) as dst:
        block_num = 0
        for y in range(0, FULL_HEIGHT, TILE_SIZE):
            for x in range(0, FULL_WIDTH, TILE_SIZE):
                block_h = min(TILE_SIZE, FULL_HEIGHT - y)
                block_w = min(TILE_SIZE, FULL_WIDTH - x)

                pred = predict_block(src, model, y, x, block_h, block_w, TILE_BUFFER)

                window = Window(x, y, block_w, block_h)
                dst.write(pred, 1, window=window)

                block_num += 1
                print(f'  Block {block_num}/{total_blocks} ({100*block_num/total_blocks:.0f}%)', end='\r')

print(f'\nRaw prediction saved: {RAW_PATH}')

RAW_DRIVE = os.path.join(DRIVE_SAVE_DIR, 'galamsey_prediction_raw.tif')
shutil.copy2(RAW_PATH, RAW_DRIVE)
print(f'Raw saved to Drive: {RAW_DRIVE}')

with rasterio.open(RAW_PATH) as src:
    total_gal = 0
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        total_gal += src.read(1, window=Window(0, y, FULL_WIDTH, h)).sum()
    print(f'Galamsey pixels: {total_gal:,}')

# Free GPU memory before post-processing
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('GPU memory released')

## Post-processing
Each step reads strips from the previous file and writes to the next.
No full raster is ever loaded into memory. Intermediate files are deleted
after each step to conserve disk space.

In [ ]:
# Count raw pixels without loading the full raster
with rasterio.open(RAW_PATH) as src:
    pred_profile = src.profile.copy()
    original_count = 0
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        original_count += int(src.read(1, window=Window(0, y, FULL_WIDTH, h)).sum())

print(f'Raw prediction: {original_count:,} pixels')

In [ ]:
# Nodata masking: zero out predictions where all image bands are zero
# (tile edges, gaps between swaths)
step1_path = os.path.join(WORK_DIR, 'pp_step1_nodata.tif')
out_prof = pred_profile.copy()
out_prof.update(dtype='uint8', count=1, compress='lzw')

removed_nodata = 0
with rasterio.open(RAW_PATH) as pred_src, \
     rasterio.open(VRT_PATH) as img_src, \
     rasterio.open(step1_path, 'w', **out_prof) as dst:
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        window = Window(0, y, FULL_WIDTH, h)
        strip = pred_src.read(1, window=window)
        bands = img_src.read(window=window)
        nodata_mask = np.all(bands == 0, axis=0)
        removed_nodata += int(((strip == 1) & nodata_mask).sum())
        strip[nodata_mask] = 0
        dst.write(strip, 1, window=window)
        del bands, nodata_mask, strip

print(f'Nodata removed: {removed_nodata:,}')

In [ ]:
# NDVI filter pass 1: sample NDVI values from predicted galamsey pixels
# to compute the Otsu threshold (separates bare ground from vegetation)
RED_BAND = 3   # rasterio 1-indexed; Planet NICFI order: B, G, R, NIR
NIR_BAND = 4

ndvi_samples = []
with rasterio.open(step1_path) as pred_src, \
     rasterio.open(VRT_PATH) as img_src:
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        window = Window(0, y, FULL_WIDTH, h)
        strip = pred_src.read(1, window=window)
        red = np.nan_to_num(img_src.read(RED_BAND, window=window).astype(np.float32))
        nir = np.nan_to_num(img_src.read(NIR_BAND, window=window).astype(np.float32))
        denom = nir + red
        ndvi = np.where(denom > 0, (nir - red) / denom, 0).astype(np.float32)
        gal_ndvi = ndvi[strip == 1]
        if len(gal_ndvi) > 0:
            if len(gal_ndvi) > 50000:
                gal_ndvi = np.random.choice(gal_ndvi, 50000, replace=False)
            ndvi_samples.append(gal_ndvi)
        del strip, red, nir, denom, ndvi

if ndvi_samples:
    ndvi_all = np.concatenate(ndvi_samples)
    ndvi_threshold = threshold_otsu(ndvi_all)
    print(f'Otsu NDVI threshold: {ndvi_threshold:.4f}  ({len(ndvi_all):,} samples)')
    del ndvi_samples, ndvi_all
else:
    ndvi_threshold = None
    print('No galamsey pixels for NDVI — skipping filter.')

In [ ]:
# NDVI filter pass 2: apply threshold, write result to new file
step2_path = os.path.join(WORK_DIR, 'pp_step2_ndvi.tif')

removed_ndvi = 0
with rasterio.open(step1_path) as pred_src, \
     rasterio.open(VRT_PATH) as img_src, \
     rasterio.open(step2_path, 'w', **out_prof) as dst:
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        window = Window(0, y, FULL_WIDTH, h)
        strip = pred_src.read(1, window=window)
        if ndvi_threshold is not None:
            red = np.nan_to_num(img_src.read(RED_BAND, window=window).astype(np.float32))
            nir = np.nan_to_num(img_src.read(NIR_BAND, window=window).astype(np.float32))
            denom = nir + red
            ndvi = np.where(denom > 0, (nir - red) / denom, 0).astype(np.float32)
            veg_mask = ndvi > ndvi_threshold
            removed_ndvi += int(((strip == 1) & veg_mask).sum())
            strip[veg_mask] = 0
            del red, nir, denom, ndvi, veg_mask
        dst.write(strip, 1, window=window)
        del strip

print(f'NDVI removed: {removed_ndvi:,}')

os.remove(step1_path)

In [ ]:
# Built-up mask: remove predictions overlapping settlements and urban areas.
# Uses gdalwarp for resampling if the mask dimensions don't match the mosaic.
step3_path = os.path.join(WORK_DIR, 'pp_step3_builtup.tif')

removed_builtup = 0

if BUILTUP_PATH and os.path.exists(BUILTUP_PATH):
    with rasterio.open(BUILTUP_PATH) as bu_src:
        bu_shape = (bu_src.height, bu_src.width)
        needs_resample = bu_shape != (FULL_HEIGHT, FULL_WIDTH)

    if needs_resample:
        print(f'Resampling built-up from {bu_shape} to ({FULL_HEIGHT}, {FULL_WIDTH})')
        bu_resampled_path = os.path.join(WORK_DIR, 'builtup_resampled.tif')
        subprocess.run([
            'gdalwarp', '-overwrite',
            '-ts', str(FULL_WIDTH), str(FULL_HEIGHT),
            '-r', 'near',
            BUILTUP_PATH, bu_resampled_path
        ], check=True)
        bu_read_path = bu_resampled_path
    else:
        bu_read_path = BUILTUP_PATH

    with rasterio.open(step2_path) as pred_src, \
         rasterio.open(bu_read_path) as bu_src, \
         rasterio.open(step3_path, 'w', **out_prof) as dst:
        for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
            h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
            window = Window(0, y, FULL_WIDTH, h)
            strip = pred_src.read(1, window=window)
            bu_strip = bu_src.read(1, window=window)
            removed_builtup += int(((strip == 1) & (bu_strip > 0)).sum())
            strip[bu_strip > 0] = 0
            dst.write(strip, 1, window=window)
            del strip, bu_strip

    print(f'Built-up removed: {removed_builtup:,}')
else:
    shutil.copy2(step2_path, step3_path)
    print('No built-up mask found, skipping.')

os.remove(step2_path)

In [ ]:
# GDAL sieve: remove isolated clusters smaller than MIN_PATCH_PIXELS.
# Operates on files directly — memory-safe and preserves crisp boundaries
# (unlike morphological opening which rounds edges).
with rasterio.open(step3_path) as src:
    presieve_count = 0
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        presieve_count += src.read(1, window=Window(0, y, FULL_WIDTH, h)).sum()
print(f'Pre-sieve pixels: {presieve_count:,}')

sieved_path = os.path.join(WORK_DIR, 'pp_step4_sieved.tif')

src_ds = gdal.Open(step3_path)
driver = gdal.GetDriverByName('GTiff')
dst_ds = driver.CreateCopy(sieved_path, src_ds, options=['COMPRESS=LZW'])
src_ds = None

dst_ds = gdal.Open(sieved_path, gdal.GA_Update)
band = dst_ds.GetRasterBand(1)
gdal.SieveFilter(band, None, band, MIN_PATCH_PIXELS, 4)  # 4-connected neighbourhood
band.FlushCache()
dst_ds = None

with rasterio.open(sieved_path) as src:
    sieved_count = 0
    for y in range(0, FULL_HEIGHT, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, FULL_HEIGHT - y)
        sieved_count += src.read(1, window=Window(0, y, FULL_WIDTH, h)).sum()

removed_noise = int(presieve_count - sieved_count)
print(f'Sieve removed: {removed_noise:,} pixels (clusters < {MIN_PATCH_PIXELS} px)')

os.remove(step3_path)

## Save and summarise

In [ ]:
CLEAN_DRIVE = os.path.join(DRIVE_SAVE_DIR, 'galamsey_prediction_cleaned.tif')
shutil.copy2(sieved_path, CLEAN_DRIVE)
print(f'Cleaned saved to Drive: {CLEAN_DRIVE}')

pixel_area = abs(pred_profile['transform'][0] * pred_profile['transform'][4])
orig_km2 = (original_count * pixel_area) / 1e6
clean_km2 = (int(sieved_count) * pixel_area) / 1e6

print(f'\n{"=" * 50}')
print(f'Post-processing summary')
print(f'{"=" * 50}')
print(f'Original:     {original_count:>12,} pixels ({orig_km2:.2f} km2)')
print(f'Nodata:       {removed_nodata:>12,} removed')
print(f'NDVI (Otsu):  {removed_ndvi:>12,} removed (threshold: {ndvi_threshold:.4f if ndvi_threshold is not None else "N/A"})')
print(f'Built-up:     {removed_builtup:>12,} removed')
print(f'Sieve:        {removed_noise:>12,} removed (< {MIN_PATCH_PIXELS} px)')
print(f'Final:        {int(sieved_count):>12,} pixels ({clean_km2:.2f} km2)')
print(f'Reduction:    {100*(1-sieved_count/max(original_count,1)):.1f}%')
print(f'\nRaw:     {RAW_DRIVE}')
print(f'Cleaned: {CLEAN_DRIVE}')

In [ ]:
# Remove local working files to free Colab disk space
# Uncomment to run
# shutil.rmtree(WORK_DIR)
# print('Working directory cleaned')